# Zepto Data & AI Platform — Module 2: Exploratory Data Analysis (EDA)
This notebook profiles the Titanic customer-style dataset, implements missing value strategies, and performs statistical univariate, bivariate, correlation, and multivariate exploratory analysis.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dataset_loader import get_or_load_titanic

df_raw = get_or_load_titanic()
print(f"Raw dataset shape: {df_raw.shape}")
df_raw.head()

## 1. Missing Value Strategy & Cleaning
Applying project threshold rules:
- `<5% missing`: Drop affected rows (`embarked`)
- `5-30% missing`: Median imputation (`age`)
- `>30% missing`: Drop column (`deck`)

In [ ]:
missing_pct = (df_raw.isnull().sum() / len(df_raw)) * 100
print("Missing percentages:\n", missing_pct[missing_pct > 0])

df_clean = df_raw.drop(columns=['deck']).dropna(subset=['embarked']).reset_index(drop=True)
print(f"Cleaned dataset shape: {df_clean.shape}")

## 2. Univariate Analysis (Age & Fare)
Histograms, box plots, IQR outlier detection, and central tendency skewness evaluation.

In [ ]:
for col in ['age', 'fare']:
    q1, q3 = df_clean[col].quantile(0.25), df_clean[col].quantile(0.75)
    iqr = q3 - q1
    outliers = df_clean[(df_clean[col] < q1 - 1.5*iqr) | (df_clean[col] > q3 + 1.5*iqr)]
    print(f"{col.upper()}: IQR={iqr:.2f}, Outliers={len(outliers)}")

print(f"Fare Mean: {df_clean['fare'].mean():.2f}, Median: {df_clean['fare'].median():.2f}, Mode: {df_clean['fare'].mode()[0]:.2f}")
# Skewness conclusion: Mean > Median > Mode indicates Right-Skewed distribution

## 3. Bivariate Survival Rates
Calculating survival rates by `sex`, `pclass`, and `sex + pclass` via boolean masks.

In [ ]:
print('Female survival:', df_clean[df_clean['sex'] == 'female']['survived'].mean())
print('Male survival:', df_clean[df_clean['sex'] == 'male']['survived'].mean())
for s in ['female', 'male']:
    for c in [1, 2, 3]:
        rate = df_clean[(df_clean['sex'] == s) & (df_clean['pclass'] == c)]['survived'].mean()
        print(f"{s} class {c} survival: {rate:.4f}")

## 4. 6-Column Correlation Analysis
Restricted to: `survived`, `pclass`, `age`, `sibsp`, `parch`, `fare`.

In [ ]:
corr_cols = ['survived', 'pclass', 'age', 'sibsp', 'parch', 'fare']
corr = df_clean[corr_cols].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('6-Variable Correlation Matrix')
plt.show()